In [1]:
# Configure Logfire & Instrument Pydantic AI
from pydantic_ai import Agent
from dotenv import load_dotenv
load_dotenv()

import logfire
logfire.configure()
logfire.instrument_pydantic_ai()
# The Logfire dashboard is available at https://logfire-eu.pydantic.dev/leotideman/starter-project

Logfire project URL: ]8;id=28234;https://logfire-eu.pydantic.dev/leotideman/starter-project\https://logfire-eu.pydantic.dev/leotideman/starter-project]8;;\


In [ ]:
# Test tools
#from trivia_tools import TriviaTools
#tools = TriviaTools()
#print(tools.get_categories())
#print(tools.get_questions(amount=3, category=9, difficulty="easy"))

In [ ]:
#from trivia_agent import agent
#result = await agent.run("Let's play 3 medium questions from History") 
#print(result.output)

In [2]:
from trivia_agent import agent
async def run(prompt):
    message_history = []

    while True:
        result = await agent.run(prompt, message_history=message_history)
        print(result.output)
        message_history = result.all_messages()

        prompt = input("You (write 'stop' to stop): ")
        if not prompt or prompt.lower().strip() == 'stop':
            break

In [ ]:
#await run("Let's play 5 easy questions from Science & Nature")

In [3]:
# Grouping Logfire traces
with logfire.span('trivia_session'):
    await run("Let's play 5 easy questions from Science & Nature")

22:32:16.783 trivia_session
22:32:16.788   agent run
22:32:16.810     chat gpt-4o-mini
22:32:18.934     running 1 tool
22:32:18.934       running tool: get_categories
22:32:19.387     chat gpt-4o-mini
22:32:21.273     running 1 tool
22:32:21.277       running tool: get_questions
22:32:21.658     chat gpt-4o-mini
Great! Let's get started with your trivia questions on Science & Nature. Here comes the first question:

### Question 1:
Which element has the highest melting point?
- A) Tungsten
- B) Platinum
- C) Carbon
- D) Osmium

Please provide your answer (A, B, C, or D)!


You (write 'stop' to stop):  Tungsten


22:32:54.607   agent run
22:32:54.607     chat gpt-4o-mini
That's incorrect. The correct answer is **C) Carbon**. 

**Interesting Fact:** Carbon has a melting point of around 3550 °C (6422 °F) when it is in the form of graphite. This makes it one of the elements with the highest melting points, and it plays a vital role in various applications, including steel production and the synthesis of advanced materials.

Your score is 0. Let's move on to the next question!

### Question 2:
Naturally occurring uranium primarily consists of which isotope?
- A) 235
- B) 238
- C) 239
- D) 233

What’s your answer?


You (write 'stop' to stop):  238


22:33:37.684   agent run
22:33:37.688     chat gpt-4o-mini
That's correct! The answer is **B) 238**.

**Interesting Fact:** About 99.3% of naturally occurring uranium is the isotope uranium-238, while only about 0.7% is uranium-235, which is the isotope used in nuclear reactors and bombs. Uranium-238 is predominantly used in the production of plutonium for nuclear weapons and energy.

Your score is now 1. Let's proceed to the next question!

### Question 3:
What is the hottest planet in the solar system?
- A) Jupiter
- B) Mercury
- C) Venus
- D) Mars

What's your answer?


You (write 'stop' to stop):  Mercury


22:33:49.139   agent run
22:33:49.142     chat gpt-4o-mini
That's incorrect. The correct answer is **C) Venus**.

**Interesting Fact:** Although Venus is the second planet from the Sun, it has a thick atmosphere rich in carbon dioxide, which creates an extreme greenhouse effect, raising surface temperatures to about 467 °C (872 °F). This makes Venus hotter than Mercury, despite being further from the Sun.

Your score remains 1. Here comes the next question!

### Question 4:
What is the thin, outermost layer of the Earth?
- A) Exosphere
- B) Mantle
- C) Crust
- D) Outer Core

What’s your answer?


You (write 'stop' to stop):  stop


In [4]:
# Querying trace data 
import os
from logfire.query_client import LogfireQueryClient

logfire_query_client = LogfireQueryClient(read_token=os.environ["LOGFIRE_READ_TOKEN"])

trace_rows = logfire_query_client.query_json_rows(
    sql="""
    SELECT
        trace_id,
        start_timestamp,
        duration
    FROM records
    WHERE span_name = 'agent run'
    ORDER BY start_timestamp DESC
    LIMIT 2
    """
)

In [5]:
trace_ids = [r['trace_id'] for r in trace_rows['rows']]
trace_ids

['019cd483da0fa7617d7c9f57df629d93', '019cd483da0fa7617d7c9f57df629d93']

In [6]:
trace_id = trace_ids[0]

run_row = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        attributes->'pydantic_ai.all_messages' as all_messages,
        attributes->>'gen_ai.usage.input_tokens' as input_tokens,
        attributes->>'gen_ai.usage.output_tokens' as output_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND span_name = 'agent run'
    ORDER BY start_timestamp DESC
    LIMIT 1
    """
)

In [7]:
all_messages = run_row['rows'][0]
all_messages['all_messages'][0]

{'role': 'user',
 'parts': [{'type': 'text',
   'content': "Let's play 5 easy questions from Science & Nature"}]}

### Reverse engineering: from Logfire (open telemetry) to Pydantic AI format
The code connects to Logfire, finds the latest agent run trace, and reconstructs it into an AgentRunResult using the trace_replay module (by Alexey Grigorev). It then gets the token usage either from the reconstructed trace and it calculates the approximate session cost using the gpt-4o-mini pricing formula and prints the result.

In [9]:
from trace_replay_converter import fetch_trace, trace_to_run_result

MODEL_PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
}

def calculate_cost(model_name, input_tokens, output_tokens):
    """
    Calculate the estimated API cost for a model call based on token usage.

    Args:
        model_name (str): The name of the model, used to look up input and output
            token prices in MODEL_PRICES.
        input_tokens (int): The number of input tokens sent to the model.
        output_tokens (int): The number of output tokens generated by the model.

    Returns:
        float: The total estimated cost in USD.
    """
    prices = MODEL_PRICES[model_name.lower()]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    return input_cost + output_cost


def get_latest_trace_id(client: LogfireQueryClient) -> str:
    """
    Fetch the most recent trace ID for an 'agent run' span from Logfire.

    Args:
        client (LogfireQueryClient): An authenticated Logfire query client.

    Returns:
        str: The trace ID of the latest matching agent run.

    Raises:
        ValueError: If no 'agent run' traces are found.
    """
    rows = client.query_json_rows(
        sql="""
        SELECT
            trace_id,
            start_timestamp
        FROM records
        WHERE span_name = 'agent run'
        ORDER BY start_timestamp DESC
        LIMIT 1
        """
    )["rows"]

    if not rows:
        raise ValueError("No 'agent run' traces found.")

    return rows[0]["trace_id"]


def get_chat_token_counts(client: LogfireQueryClient, trace_id: str) -> tuple[int, int]:
    """
    Sum input and output token usage for all chat-related spans in a trace.

    Args:
        client (LogfireQueryClient): An authenticated Logfire query client.
        trace_id (str): The trace ID whose chat spans should be queried.

    Returns:
        tuple[int, int]: A pair of (input_tokens, output_tokens) aggregated
        across all matching chat spans.
    """
    rows = client.query_json_rows(
        sql=f"""
        SELECT
            attributes->>'gen_ai.usage.input_tokens' AS input_tokens,
            attributes->>'gen_ai.usage.output_tokens' AS output_tokens
        FROM records
        WHERE trace_id = '{trace_id}'
          AND span_name LIKE 'chat%'
        ORDER BY start_timestamp ASC
        """
    )["rows"]

    input_tokens = sum(int(row.get("input_tokens") or 0) for row in rows)
    output_tokens = sum(int(row.get("output_tokens") or 0) for row in rows)

    return input_tokens, output_tokens


with LogfireQueryClient(read_token=os.environ["LOGFIRE_READ_TOKEN"]) as client:
    # If you already know the trace id, replace this with:
    # trace_id = "your-trace-id"
    trace_id = get_latest_trace_id(client)

    # Rebuild AgentRunResult from the lesson module
    trace = fetch_trace(trace_id, client)
    result = trace_to_run_result(trace)

    # Prefer token counts from the reconstructed trace
    input_tokens = trace.input_tokens
    output_tokens = trace.output_tokens

    total_cost = calculate_cost("gpt-4o-mini", input_tokens, output_tokens)

In [10]:
print("trace_id:", trace_id)
print("result.output:", result.output)
print("input_tokens:", input_tokens)
print("output_tokens:", output_tokens)
print("total_cost_usd:", total_cost)
print("total_cost_usd_rounded:", round(total_cost, 8))

trace_id: 019cd483da0fa7617d7c9f57df629d93
result.output: That's incorrect. The correct answer is **C) Venus**.

**Interesting Fact:** Although Venus is the second planet from the Sun, it has a thick atmosphere rich in carbon dioxide, which creates an extreme greenhouse effect, raising surface temperatures to about 467 °C (872 °F). This makes Venus hotter than Mercury, despite being further from the Sun.

Your score remains 1. Here comes the next question!

### Question 4:
What is the thin, outermost layer of the Earth?
- A) Exosphere
- B) Mantle
- C) Crust
- D) Outer Core

What’s your answer?
input_tokens: 928
output_tokens: 131
total_cost_usd: 0.00021779999999999998
total_cost_usd_rounded: 0.0002178
